# BLIP 画像キャプション生成サンプル

## BLIP — 画像キャプション生成

このノートブックは **BLIP（Bootstrapping Language-Image Pre-training）** を使った画像キャプション生成のサンプルです。

```
画像（URL またはファイル）→ BLIP モデル → 自然言語キャプション
```

BLIP は画像全体を説明する自然言語テキストを生成します。写っている物体・動作・シーン全体のコンテキストを文章として出力します。

### 使用可能なモデル

| モデル | 説明 | 速度 |
|-------|------|------|
| `Salesforce/blip-image-captioning-large` | より詳細なキャプション | 遅い |
| `Salesforce/blip-image-captioning-base`  | 高品質・高速 | 速い |

writefile セル内の `MODEL_NAME` を変更することでモデルを切り替えられます。

📄 [BLIP 論文](https://arxiv.org/abs/2201.12086)  
🤗 [Hugging Face モデルハブ](https://huggingface.co/Salesforce/blip-image-captioning-large)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/blip"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Download sample images from GitHub
import os

BASE_URL = "https://raw.githubusercontent.com/mastnk/cv-samples/main/blip"
IMAGE_FILES = [
    "furgao52-monkey-9040130_640.jpg",
]
for fname in IMAGE_FILES:
    url  = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Install dependencies
!pip install transformers accelerate -q

import transformers
print(transformers.__version__)  # check version


## 独自画像を使うには

画像を用意する方法は 2 つあります。

**① URL を指定する**  
スクリプト実行時に `--url` フラグに直接画像 URL を渡します：
```
%run blip.py --url https://cdn.pixabay.com/photo/2022/09/09/03/42/thailand-7442285_640.jpg
```

**② 画像ファイルを `PROJECT_PATH/` に置く**  
画像ファイルを `PROJECT_PATH/` に配置してから `--file` または `--dir` で実行します。

アップロードは **Google Drive** 経由が簡単です：  
[drive.google.com](https://drive.google.com) を開き、`マイドライブ / CV-Samples / blip/` に移動してファイルをドラッグ＆ドロップするだけ。  
マウント済みの Drive を通じて Colab から即座にアクセスできます。追加のアップロード手順は不要です。

```
%run blip.py --file your_image.jpg   # ファイル 1 枚
%run blip.py --dir  .                # フォルダ内の全画像
```

## モデルを選択するには

下の writefile セル内の `MODEL_NAME` を変更してモデルを切り替えます。  
`MODEL_NAME = ...` の行が複数ある場合、**有効になるのは最後の 1 行だけ**です。

```python
# MODEL_NAME = 'Salesforce/blip-image-captioning-large'  # more descriptive, slower
MODEL_NAME   = 'Salesforce/blip-image-captioning-base'   # faster, good quality  ← active
```

`large` モデルはより豊かなキャプションを生成しますが、ロードと実行に時間がかかります。

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile blip.py
"""BLIP Image Captioning — command-line interface.

Usage:
  %run blip.py --url  <image_url>  [--disp]
  %run blip.py --file <image_path> [--disp]
  %run blip.py --dir  <image_dir>  [--disp]
"""

import argparse
import glob
import os

from PIL import Image
import requests
from io import BytesIO
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME = 'Salesforce/blip-image-captioning-base'   # faster, good quality
# MODEL_NAME = 'Salesforce/blip-image-captioning-large'  # more descriptive, slower

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/blip'

# ---- Model loading -----------------------------------------------
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model     = BlipForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    tie_word_embeddings=False,
).to(device)

# ---- Caption helper ----------------------------------------------
def generate_caption(image: Image.Image) -> str:
    """Generate a caption for a PIL image."""
    inputs = processor(image, return_tensors='pt').to(device)
    out    = model.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)

# ---- Display helper ----------------------------------------------
def show(image: Image.Image, label: str, disp: bool) -> None:
    """When --disp is active, display the image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(image)
        print(label)

# ---- Functions ---------------------------------------------------
def caption_url(url: str, disp: bool = False):
    """Download an image from a URL and generate a caption."""
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    show(image, url, disp)
    caption = generate_caption(image)
    print(f'  Caption: {caption}')
    return caption


def caption_file(path: str, disp: bool = False):
    """Generate a caption for a single local image file."""
    image   = Image.open(path).convert('RGB')
    show(image, path, disp)
    caption = generate_caption(image)
    print(f'  Caption: {caption}')
    return caption


def caption_dir(directory: str, disp: bool = False):
    """Generate captions for all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        caption_file(path, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='BLIP Image Captioning')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL to caption')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images to caption')
parser.add_argument('--disp', action='store_true',
                    help='Display image and filename/URL before caption')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    caption_url(args.url,   disp=args.disp)
elif args.file:
    caption_file(args.file, disp=args.disp)
elif args.dir:
    caption_dir(args.dir,   disp=args.disp)


## `blip.py` の使い方

上の `%%writefile` セルを実行すると、`blip.py` が `PROJECT_PATH` に保存されます。  
`--disp` でのインライン画像表示を有効にするため、`!python` ではなく **`%run`** で実行してください。

```
%run blip.py --url  <画像URL>        # リモート画像のキャプション生成
%run blip.py --file <画像パス>       # ローカルファイル 1 枚のキャプション生成
%run blip.py --dir  <ディレクトリ>  # フォルダ内の全画像のキャプション生成
```

**オプション引数**

| フラグ | デフォルト | 説明 |
|--------|-----------|------|
| `--disp` | オフ | 結果の前に画像とファイル名 / URL を表示 |

**実行例**

```bash
# URL からキャプション生成（画像表示あり）
%run blip.py --url https://cdn.pixabay.com/photo/2022/09/09/03/42/thailand-7442285_640.jpg --disp

# ローカルファイルのキャプション生成
%run blip.py --file furgao52-monkey-9040130_640.jpg --disp

# フォルダ内の全画像のキャプション生成
%run blip.py --dir . --disp
```

**出力形式（`--disp` あり）**

```
<画像をインライン表示>
furgao52-monkey-9040130_640.jpg
  Caption: a monkey sitting on a branch in a tree
```

## 実行方法

`blip.py` は `!python` ではなく **`%run`** で実行してください。Colab カーネル内で実行されるため、`--disp` でのインライン画像表示が有効になります。

| モード | フラグ | 説明 |
|--------|--------|------|
| URL から | `--url <url>` | リモート画像を取得してキャプション生成 |
| ファイル 1 枚 | `--file <パス>` | ローカル画像 1 枚のキャプション生成 |
| ディレクトリ | `--dir <パス>` | フォルダ内の全画像のキャプション生成 |

`--disp` を追加すると、キャプションの前に各画像とファイル名 / URL が表示されます。

In [ ]:
# Execute: generate caption from URL (with display)
%cd "{PROJECT_PATH}"
%run blip.py --disp --url https://cdn.pixabay.com/photo/2022/09/09/03/42/thailand-7442285_640.jpg


In [ ]:
# Execute: generate caption from a single local image file (with display)
%cd "{PROJECT_PATH}"
%run blip.py --disp --file furgao52-monkey-9040130_640.jpg


In [ ]:
# Execute: generate captions for all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run blip.py --disp --dir .


💾 **ノートブックを保存するのを忘れずに！**

作業内容を保持するため、閉じる前に **ファイル → ドライブにコピーを保存** を実行してください。